In [1]:
!apt update
!apt install unzip

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1581 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy InRelease [270 kB]                
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2597 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]3m
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:8 http://archive.ubuntu.com/ubuntu jammy/multiverse amd64 Packages [266 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy/restricted amd64 Packages [164 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy/universe amd64 Packages [17.5 MB]
Get:11 http://archive.ubuntu.com/ubuntu jammy/main amd64 Packages [1792 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Pa

In [2]:
!unzip data2.zip

Archive:  data2.zip
   creating: labeler/train/
   creating: labeler/train/images/
  inflating: labeler/train/images/vid45_002_jpg.rf.QywdGvV5HgMVZ1BvJI5Z.jpg  
  inflating: labeler/train/images/vid30_002_jpg.rf.wwTOohdNpSUujMtGEBTn.jpg  
  inflating: labeler/train/images/frame_016_jpg.rf.NLV0jUrfuIoaVKuXhDxb.jpg  
  inflating: labeler/train/images/frame_022_jpg.rf.rKGon0JnZA5urvQhdyt5.jpg  
  inflating: labeler/train/images/frame_056_jpg.rf.yOxArH2WQKIdnIQ8ErZ8.jpg  
  inflating: labeler/train/images/frame_052_jpg.rf.pe6y05U1sAzDl08ruo8Q.jpg  
  inflating: labeler/train/images/vid41_005_jpg.rf.3iKuSR1S0EIVqJi82tO9.jpg  
  inflating: labeler/train/images/vid39_000_jpg.rf.XT3NTFYTnz7oTuwaLIzK.jpg  
  inflating: labeler/train/images/vid27_003_jpg.rf.rlPLLIxLkhMq7xGHoC6a.jpg  
  inflating: labeler/train/images/vid40_009_jpg.rf.Nxp6TiFj3luuv5riWuKe.jpg  
  inflating: labeler/train/images/frame_062_jpg.rf.HF6XB6KDXpWwVQc0k1Bk.jpg  
  inflating: labeler/train/images/vid46_014_jpg.rf.D6cuvZat

In [3]:
!find . -type f -name "*Identifier*" -delete

In [5]:
# 데이터셋 개수 확인
!ls /workspace/labeler/train/labels | wc -l
!ls /workspace/labeler/train/images | wc -l
!ls /workspace/labeler/valid/labels | wc -l
!ls /workspace/labeler/valid/images | wc -l

225
225
34
34


In [6]:
!pip install ultralytics
!pip install Pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 131.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 161.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 151.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.7/828.7 kB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 211.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 177.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 204.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 92.5 MB/s eta 0:00:00
  Attempting uninstall: pyparsing
    Found existing installation: pyparsing 2.4.7
    Uninstalling pyparsing-2.4.7:
      Successfully uninstalled pyparsing-2.4.7
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.3
    Uninstalling numpy-1.26.3:


In [13]:
# 학습
from ultralytics import YOLO

# 1. 모델 로드 (v11 나노 모델)
model = YOLO('yolo11n.pt')

# 2. 학습 시작
results = model.train(
    data='labeler.yaml',
    epochs=500,
    imgsz=640,            # 640 고정 (원본 해상도와 찰떡궁합)
    batch=64,
    patience=100,          # 과적합 방지를 위한 조기 종료 (필수)
    
    # --- [핵심 수정] 원본 크기 훼손 방지 ---
    mosaic=0.1,           # 0.5 -> 0.1: 아예 0으로 끄거나 아주 가끔만 작동하게 만듭니다. (가장 중요)
    scale=0.0,            # 0.2 -> 0.0: 이미지를 축소/확대하지 말고 원본 크기 그대로 학습시킵니다.
    mixup=0.0,            # 이미지가 겹쳐서 흐려지는 현상 차단
    copy_paste=0.0,       # 모자이크를 끄면 copy_paste도 끄는 것이 충돌을 막습니다.
    
    # --- 분류 가중치 집중 ---
    box=10.0,
    cls=3.0,              # 분류 가중치는 높게 유지

    hsv_h=0.015,   # 빨강/초록의 정체성은 지키되 약간의 톤 변화만 허용
    hsv_s=0.7,     # 쨍한 불빛부터 빛바랜 불빛까지 모두 대응
    hsv_v=0.4,     # 역광이나 그늘진 환경 대응

    translate=0.1,  # 이미지 내에서 신호등의 위치를 상하좌우로 10% 정도 이동 (위치에 대한 과적합 방지)
    fliplr=0.5,     # 50% 확률로 좌우 반전 (신호등은 좌우가 뒤집혀도 정체성이 변하지 않으므로 데이터 2배 뻥튀기 효과)
    erasing=0.1,    # 10% 확률로 이미지의 일부를 가림 (나뭇가지나 표지판에 신호등이 살짝 가려지는 상황 대비)
)

Ultralytics 8.4.43 🚀 Python-3.11.10 torch-2.4.1+cu124 CUDA:0 (NVIDIA GeForce RTX 4090, 24081MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=10.0, cache=False, cfg=None, classes=None, close_mosaic=10, cls=3.0, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=labeler.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=500, erasing=0.1, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=0.1, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, 

In [12]:
!rm -rf runs